<a href="https://colab.research.google.com/github/andreelzs/Linguagens-de-programacao/blob/main/sistema_rh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de RH com SQLAlchemy

In [1]:
!pip install -q sqlalchemy pandas

import pandas as pd
from sqlalchemy import (
    create_engine, text, MetaData, Table, Column,
    Integer, String, Float, ForeignKey, insert, update, select, func
)
from sqlalchemy.orm import (
    declarative_base, relationship, mapped_column, Mapped, sessionmaker
)

## Nível 1: Básico — Configuração e SQL Puro com Segurança

In [2]:
# Passo 1: conexão
engine = create_engine('sqlite:///sistema_rh.db')

In [3]:
# Passo 2: criação da tabela via SQL puro
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    """))

In [4]:
# Passo 3: inserção segura com parâmetros (evita SQL Injection)
novo_funcionario = {'nome': 'Beatriz Nunes', 'cargo': 'Desenvolvedor Júnior', 'salario': 3500.00}

with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
        novo_funcionario
    )

In [5]:
# Passo 4: validação com pandas
df_funcionarios = pd.read_sql_query("SELECT * FROM funcionarios", engine)
df_funcionarios

,id,nome,cargo,salario
0,1,Beatriz Nunes,Desenvolvedor Júnior,3500.0


## Nível 2: Intermediário — SQLAlchemy Core

In [6]:
# Passo 1: tabela projetos definida programaticamente
metadata = MetaData()

projetos = Table(
    'projetos', metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome_projeto', String, nullable=False),
    Column('funcionario_id', Integer, nullable=False),
    Column('orcamento', Float, nullable=False),
)

metadata.create_all(engine)

In [7]:
# Passo 2: bulk insert
lista_projetos = [
    {'nome_projeto': 'Portal RH', 'funcionario_id': 1, 'orcamento': 15000.00},
    {'nome_projeto': 'App Ponto', 'funcionario_id': 1, 'orcamento': 8000.00},
]

with engine.begin() as conn:
    conn.execute(insert(projetos), lista_projetos)

In [8]:
# Passo 3: reajuste salarial dos Desenvolvedor Júnior
funcionarios_tbl = Table('funcionarios', metadata, autoload_with=engine)

with engine.begin() as conn:
    conn.execute(
        update(funcionarios_tbl)
        .where(funcionarios_tbl.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios_tbl.c.salario * 1.10)
    )

In [9]:
# Passo 4: relatório salarial agregado por cargo
with engine.connect() as conn:
    resultado = conn.execute(
        select(
            funcionarios_tbl.c.cargo,
            func.avg(funcionarios_tbl.c.salario).label('salario_medio')
        ).group_by(funcionarios_tbl.c.cargo)
    )
    for linha in resultado:
        print(linha.cargo, linha.salario_medio)

Desenvolvedor Júnior 3850.0000000000005


## Nível 3: Avançado — ORM

In [10]:
# Passo 1 e 2: classes ORM com relacionamento
Base = declarative_base()


class Departamento(Base):
    __tablename__ = 'departamentos'

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)

    funcionarios: Mapped[list['FuncionarioORM']] = relationship(back_populates='departamento')


class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)
    cargo: Mapped[str] = mapped_column(String, nullable=False)
    salario: Mapped[float] = mapped_column(Float, nullable=False)
    departamento_id: Mapped[int] = mapped_column(ForeignKey('departamentos.id'))

    departamento: Mapped['Departamento'] = relationship(back_populates='funcionarios')


Base.metadata.create_all(engine)

In [11]:
# Passo 3: sessão, criação de objetos e persistência
SessionLocal = sessionmaker(bind=engine)
sessao = SessionLocal()

dept_ti = Departamento(nome='TI')
dept_ti.funcionarios.append(FuncionarioORM(nome='Rafael Costa', cargo='Analista de Sistemas', salario=6000.00))
dept_ti.funcionarios.append(FuncionarioORM(nome='Camila Alves', cargo='Desenvolvedor Júnior', salario=3850.00))

sessao.add(dept_ti)
sessao.commit()

In [12]:
# Passo 4: consulta orientada a objetos
stmt = (
    select(FuncionarioORM)
    .join(Departamento)
    .where(Departamento.nome == 'TI')
)

funcionarios_ti = sessao.execute(stmt).scalars().all()

for f in funcionarios_ti:
    print(f.id, f.nome, f.cargo, f.salario, f.departamento.nome)

sessao.close()

1 Rafael Costa Analista de Sistemas 6000.0 TI
2 Camila Alves Desenvolvedor Júnior 3850.0 TI
